[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/mirror_energy/mirror_energy.ipynb)

# Energy in a mirror

Kinetic energy measures motion and stays positive when a moving system is mirrored. Reflecting its inertia or rebuilding inertia from its reflected masses must give the same energy.

Spherical geometry in the three-generator algebra `x+y+z+` represents positions by antivectors and motion rates by bivectors. Joining a position to its mass-weighted velocity gives momentum; pairing momentum with the rate gives twice the kinetic energy.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import Spherical3D
from examples.animation import show_animation
from examples.orientation.mirror_energy import render

np.set_printoptions(precision=3, suppress=True)

In [ ]:
ga = Spherical3D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Point = ga.gatype.antivector()                # Antivector-
Plane = ga.gatype.vector()                    # Vector+
Rate = ga.gatype.bivector()                   # Bivector+
Momentum = ga.gatype.antibivector()           # Antibivector-
Inertia = ga.gatype((Momentum, Rate))         # Antibivector- <- Bivector+
Scalar = ga.gatype.scalar()                   # Scalar+

In [ ]:
points: Point = mv.antivector([[1, 0, 0], [0, 0.8, 0.6], [-0.6, 0, -0.8]])  # Antivector-
masses = np.array([1, 1.5, 0.5])
rate: Rate = mv.xy                                        # Bivector+
mirror: Plane = mv.x                                      # Vector+

# Inertia maps the body's rate of motion to its momentum.
inertia: Inertia = (points & points.commutator(Rate) * masses).sum(axis=0)  # Antibivector- <- Bivector+
momentum: Momentum = inertia(rate)                        # Antibivector-
energy: Scalar = (momentum & rate) / 2                    # Scalar+

print("Energy")
print(energy.coefficient_table())

The prescribed rotation carries three masses around the sphere. The right-hand motion is the reflection of the left-hand motion. Colours identify matching masses, marker sizes show their masses, and arrows show their velocities. Both have the same kinetic energy.

In [ ]:
frames = 36
duration_ms = 90
times = np.linspace(0, 2 * np.pi, frames, endpoint=False)

# Reflect the motion and both slots of the inertia map.
reflected_rate: Rate = mirror >> rate                     # Bivector+
reflected_inertia: Inertia = mirror >> inertia(mirror << Rate)  # Antibivector- <- Bivector+
reflected_energy: Scalar = (reflected_inertia(reflected_rate) & reflected_rate) / 2  # Scalar+

# Building inertia from the reflected masses gives the same result.
reflected_points: Point = mirror >> points                # Antivector-
rebuilt: Inertia = (reflected_points & reflected_points.commutator(Rate) * masses).sum(axis=0)  # Antibivector- <- Bivector+
rebuilt_energy: Scalar = (rebuilt(reflected_rate) & reflected_rate) / 2  # Scalar+

# Sample the prescribed turn and its mirror; the commutator gives tangent velocities.
positions: Point = (-rate * times[:, None] / 2).exp() >> points  # Antivector-
velocities: Point = positions.commutator(rate)            # Antivector-
paths: Point = stack([positions, mirror >> positions], axis=1)  # Antivector-
tangents: Point = stack([velocities, mirror >> velocities], axis=1)  # Antivector-

print("Reflected energy")
print(reflected_energy.coefficient_table())
print("Rebuilt energy")
print(rebuilt_energy.coefficient_table())
show_animation(render.animate_motion(paths, tangents, masses), "mirror_energy", duration_ms)

Applying grade-involuted conjugation to every quantity assigns momentum the wrong sign. Momentum has intrinsic orientation because it was joined from positions and velocities; the motion rate has extrinsic orientation. Their different grades do not encode this distinction. Applying the same grade-based reflection to both produces negative kinetic energy.

In [ ]:
# Applying grade involution uniformly discards the distinction between the two orientations.
untyped_momentum: Momentum = mirror * momentum.involute() * mirror.inverse()  # Antibivector-
untyped_rate: Rate = mirror * rate.involute() * mirror.inverse()  # Bivector+
untyped_energy: Scalar = (untyped_momentum & untyped_rate) / 2  # Scalar+

print("Energy without the orientation distinction")
print(untyped_energy.coefficient_table())